# Cross-Validation, Hyperparameter Optimization, and AutoML - Guided Practice

## Phishing Website Detection

In this session, we tackle a practical model-selection challenge: **detecting phishing websites**. We will go beyond basic model fitting and explore:

1.  **Exploratory Data Analysis (EDA)**: Understanding our cyber-threat data.
2.  **Common Pitfalls**: why high accuracy might be a lie (Accuracy Paradox & Data Leakage).
3.  **Robust Evaluation**: Cross-Validation strategies.
4.  **Hyperparameter Optimization**: Grid, Random, and Bayesian Search.
5.  **AutoML**: Automating the defense pipeline with H2O.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, GridSearchCV, RandomizedSearchCV, train_test_split, validation_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.stats import randint
import warnings
import ssl

# Handle macOS SSL certificate issues for OpenML fetch
ssl._create_default_https_context = ssl._create_unverified_context

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 1. Dataset: Phishing Websites

We use the **Phishing Websites** dataset (OpenML ID 4534). It contains features extracted from URLs (e.g., URL length, having '@' symbol, prefix/suffix usage) to classify them as legitimate or phishing.

In [ ]:
print("Loading Phishing Websites dataset...")
# OpenML ID 4534: Phishing Websites
phishing_data = fetch_openml(data_id=4534, as_frame=True, parser='auto')
df = phishing_data.frame

# Preprocessing: The dataset sometimes has -1/1 labels or similar. Let's inspect targets.
print(f"Original Target Values: {df['Result'].unique()}")

# Map to 0 (Benign) and 1 (Phishing) if needed, or ensuring they are numeric
# Often simple transformation: 1 -> Phishing, -1 -> Legitimate
# Let's assume standard LabelEncoding for consistency
le = LabelEncoder()
df['Result'] = le.fit_transform(df['Result'])

X = df.drop(columns=['Result'])
y = df['Result']

print(f"Dataset Shape: {df.shape}")
df.head()

### 1.1 Enhanced EDA
Visualizing the threat landscape.

In [ ]:
# 1. Class Distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='Result', data=df, palette='viridis')
plt.title('Distribution of Legitimate (0) vs Phishing (1) Sites')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

# 2. Correlation Heatmap
# We select a subset of numeric-like features for clarity
# Most features here are {-1, 0, 1}, so correlation is valid
plt.figure(figsize=(12, 10))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', linewidths=0.5, fmt=".1f")
plt.title('Feature Correlation Heatmap')
plt.show()

## 2. Common Pitfalls in ML Evaluations

Before we optimize, let's see how we can fool ourselves.

### 2.1 The Accuracy Paradox
If you have 95% legitimate sites and 5% phishing, a model that says "Everything is Safe" has 95% accuracy but 0% utility.
Let's simulate a highly imbalanced scenario to demonstrate.

In [ ]:
# Create an imbalanced subset
mask_majority = y == 0
mask_minority = y == 1

# Take all majority, but only 5% of minority
X_imbal = pd.concat([X[mask_majority], X[mask_minority].sample(frac=0.05, random_state=42)])
y_imbal = pd.concat([y[mask_majority], y[mask_minority].sample(frac=0.05, random_state=42)])

print(f"Imbalanced Distribution:\n{y_imbal.value_counts(normalize=True)}")

# Train a model
X_tr, X_te, y_tr, y_te = train_test_split(X_imbal, y_imbal, test_size=0.3, random_state=42)
dummy_model = RandomForestClassifier(random_state=42).fit(X_tr, y_tr)
preds = dummy_model.predict(X_te)

print(f"\nAccuracy on Imbalanced Data: {accuracy_score(y_te, preds):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_te, preds))
print("Notice how high accuracy masks the fact that we might be missing the few phishing cases!")

### 2.2 Data Leakage (The Golden Feature)
What happens if we accidentally include a feature that is measuring the target itself? Or a feature that isn't available at prediction time?

In [ ]:
# Let's create a 'leaky' feature
# Imagine a feature 'user_reported_phishing' which only exists AFTER a confirmed report
# It highly correlates with the target
X_leaky = X.copy()
# Add noise so it's not 100% perfect, but suspiciously good
X_leaky['leaky_feature'] = y + np.random.normal(0, 0.1, size=len(y))

scores_leaky = cross_val_score(RandomForestClassifier(), X_leaky, y, cv=5)
scores_clean = cross_val_score(RandomForestClassifier(), X, y, cv=5)

print(f"CV Score with Leakage: {scores_leaky.mean():.4f} (Suspicious!)")
print(f"CV Score without Leakage: {scores_clean.mean():.4f} (Realistic)")

## 3. Robust Evaluation: Cross-Validation

We return to the full, original dataset. Is `StratifiedKFold` really better?

In [ ]:
model = RandomForestClassifier(n_estimators=50, random_state=42)

# Standard K-Fold
kf = KFold(n_splits=10, shuffle=True, random_state=42)
kf_scores = cross_val_score(model, X, y, cv=kf, scoring='accuracy')

# Stratified K-Fold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
skf_scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')

print(f"Standard CV: {kf_scores.mean():.4f} (+/- {kf_scores.std()*2:.4f})")
print(f"Stratified CV: {skf_scores.mean():.4f} (+/- {skf_scores.std()*2:.4f})")

# Plotting the comparison
plt.figure(figsize=(8, 5))
plt.plot(kf_scores, label='Standard K-Fold', marker='o')
plt.plot(skf_scores, label='Stratified K-Fold', marker='x')
plt.title('CV Scores per Fold')
plt.ylabel('Accuracy')
plt.xlabel('Fold Index')
plt.legend()
plt.show()

### 3.1 Validation Curve
Visualizing overfitting/underfitting as we change a hyperparameter (e.g., Tree Depth).

In [ ]:
param_range = np.arange(1, 21, 2)
train_scores, test_scores = validation_curve(
    RandomForestClassifier(n_estimators=20, random_state=42),
    X, y, param_name="max_depth", param_range=param_range,
    cv=3, scoring="accuracy", n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(param_range, train_mean, label="Training score", color="darkorange", marker='o')
plt.plot(param_range, test_mean, label="Cross-validation score", color="navy", marker='s')
plt.title("Validation Curve with Random Forest (Max Depth)")
plt.xlabel("Max Depth")
plt.ylabel("Accuracy")
plt.legend(loc="best")
plt.show()

## 4. Hyperparameter Optimization

Searching for the best defense model.

In [ ]:
# Split hold-out set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
# 1. Randomized Search (Faster & Often Sufficient)
param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(5, 30),
    'min_samples_split': randint(2, 10),
    'max_features': ['sqrt', 'log2']
}

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist, n_iter=15, cv=3, scoring='accuracy', n_jobs=-1, random_state=42
)

print("Running Random Search...")
%time random_search.fit(X_train, y_train)
print(f"Best Random Params: {random_search.best_params_}")
print(f"Best Random Score: {random_search.best_score_:.4f}")

## 5. AutoML: The Automated Defense System (H2O)

Can an automated system beat our manually tuned forests?

In [ ]:
import h2o
from h2o.automl import H2OAutoML

try:
    h2o.init()
    
    # Convert to H2O Frame
    train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
    test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)
    
    hf_train = h2o.H2OFrame(train_df)
    hf_test = h2o.H2OFrame(test_df)

    # Define targets
    y_col = 'Result' # Ensure this matches your column name after reset_index or original
    x_cols = hf_train.columns
    x_cols.remove(y_col)
    
    # Factorize target for classification
    hf_train[y_col] = hf_train[y_col].asfactor()
    hf_test[y_col] = hf_test[y_col].asfactor()

    # Run AutoML (Limited time for demo)
    print("Running H2O AutoML...")
    aml = H2OAutoML(max_models=5, seed=1, max_runtime_secs=300, project_name='phishing_automl')
    aml.train(x=x_cols, y=y_col, training_frame=hf_train)

    # Leaderboard
    print("Leaderboard:")
    print(aml.leaderboard.head())

    # Evaluation
    preds = aml.leader.predict(hf_test)
    # Combine preds with actuals for manual inspection if needed
    # (H2O evaluation metrics are usually preferred locally)
    
except Exception as e:
    print(f"H2O Error (Java installed?): {e}")